# TT/MPS基礎 12 — TT-rounding：Right-Canonicalization と Truncated SVD Sweep

## 今回の位置づけ

Notebook 11 では、密テンソルから TT を新規構築する **TT-SVD** について、

$$
\|A-T\|_F
\le
\left(
\sum_{k=1}^{d-1}
\varepsilon_k^2
\right)^{1/2}
$$

という誤差上界と準最適性を確認しました。

今回は入力が異なります。

すでに TT cores、

$$
G^{(1)},G^{(2)},\ldots,G^{(d)}
$$

として表されているテンソル

$$
\mathcal X
$$

を、dense tensor へ戻さずに再圧縮します。

これが **TT-rounding** です。

標準的な流れは、

$$
\boxed{
\text{right-to-left QR canonicalization}
\rightarrow
\text{left-to-right truncated SVD}
}
$$

です。

### 今回の最終目標

入力 TT $\mathcal X$ から、より小さい TT-rank を持つ

$$
\widetilde{\mathcal X}
$$

を作り、指定した全体相対誤差

$$
\varepsilon
$$

に対して、

$$
\boxed{
\|\mathcal X-\widetilde{\mathcal X}\|_F
\le
\varepsilon
\|\mathcal X\|_F
}
$$

を小規模な dense reconstruction で確認します。

### 今回やること

1. 既存 TT cores の shape と rank を確認する
2. 検証用に小規模 TT を dense tensor へ戻す
3. 右→左 QR sweep で right-canonical form を作る
4. QR sweep が表すテンソルを変えないことを確認する
5. right-canonical TT では
   $$
   \|\mathcal X\|_F=\|G^{(1)}\|_F
   $$
   となることを確認する
6. 全体相対誤差 $\varepsilon$ から局所予算 $\delta$ を作る
7. 特異値の尾部エネルギーから保持 rank を決める
8. 左→右 truncated SVD sweep で TT-rank を削減する
9. 各 bond の実際の切り捨て誤差 $e_k$ を記録する
10. dense error と
    $$
    \sqrt{\sum_k e_k^2}
    $$
    を比較する

### 今回はまだ扱わないもの

- rank cap
- absolute tolerance
- TT の加算・Hadamard積
- TT-matrix / MPO
- NN weight の TT 圧縮
- DMRG
- 大規模 TT の dense reconstruction


## 1. TT-SVD と TT-rounding の違い

### TT-SVD

入力は dense tensor、

$$
\mathcal X
\in
\mathbb R^{n_1\times\cdots\times n_d}
$$

です。

dense tensor の unfolding を左→右に SVD し、

$$
G^{(1)},\ldots,G^{(d)}
$$

を新規構築します。

### TT-rounding

入力はすでに存在する TT cores、

$$
G^{(k)}
\in
\mathbb R^{r_{k-1}\times n_k\times r_k},
\qquad
r_0=r_d=1
$$

です。

dense tensor を作り直すのではなく、各 core を局所的に QR / SVD しながら、

$$
(r_1,\ldots,r_{d-1})
\longrightarrow
(\widetilde r_1,\ldots,\widetilde r_{d-1})
$$

と内部 rank を削減します。

重要なのは、

$$
(n_1,\ldots,n_d)
$$

という **物理 shape は変わらない**ことです。

変化するのは TT 内部の bond dimension です。


## 2. TT core の shape とパラメータ数

一般の第 $k$ core は、

$$
G^{(k)}
\in
\mathbb R^{r_{k-1}\times n_k\times r_k}
$$

です。

TT 全体の保存パラメータ数は、

$$
\boxed{
N_{\mathrm{TT}}
=
\sum_{k=1}^{d}
r_{k-1}n_kr_k
}
$$

です。

したがって、物理 shape を変えなくても TT-rank が下がれば、

- 保存量
- TT のまま行う縮約
- 内積
- 線形演算
- 後続 TT 演算

のコストを減らせます。

今回の学習用入力は、

$$
(n_1,n_2,n_3)=(4,5,3)
$$

とします。

まず真の内部 rank が小さい TT を作り、それを大きな bond shape へゼロ padding して、

$$
(r_1,r_2)=(4,3)
$$

という意図的に冗長な TT 表現を作ります。

これにより、rounding で不要な bond dimension を削れる状況を確実に用意します。


## 3. Right-Canonical 用と SVD 用の2種類の行列化

同じ core、

$$
G^{(k)}
\in
\mathbb R^{r_{k-1}\times n_k\times r_k}
$$

を、目的に応じて2通りに reshape します。

### 3.1 Right-canonicalization 用

左 bond を行、物理添字と右 bond を列へまとめます。

$$
H_k
=
\operatorname{reshape}
\left(
G^{(k)},
(r_{k-1},n_kr_k)
\right).
$$

shape は、

$$
H_k
\in
\mathbb R^{r_{k-1}\times(n_kr_k)}.
$$

right-canonical 条件は、

$$
\boxed{
H_kH_k^T
=
I_{r_{k-1}}
}
$$

です。

### 3.2 Left-to-right SVD 用

左 bond と物理添字を行、右 bond を列へまとめます。

$$
V_k
=
\operatorname{reshape}
\left(
G^{(k)},
(r_{k-1}n_k,r_k)
\right).
$$

shape は、

$$
V_k
\in
\mathbb R^{(r_{k-1}n_k)\times r_k}.
$$

この $V_k$ に truncated SVD を適用します。


## 4. Right-to-Left QR Sweep

処理順は、

$$
k=d,d-1,\ldots,2
$$

です。

第 $k$ core から、

$$
H_k
\in
\mathbb R^{r_{k-1}\times(n_kr_k)}
$$

を作ります。

行を正規直交化したいので、転置して reduced QR を使います。

$$
H_k^T
=
Q_kR_k.
$$

ここで、

$$
q_k
=
\min(r_{k-1},n_kr_k)
$$

とすると、

$$
Q_k
\in
\mathbb R^{(n_kr_k)\times q_k},
$$

$$
R_k
\in
\mathbb R^{q_k\times r_{k-1}}.
$$

新しい第 $k$ core は、

$$
\widehat H_k
=
Q_k^T
\in
\mathbb R^{q_k\times(n_kr_k)}
$$

を、

$$
\widehat G^{(k)}
\in
\mathbb R^{q_k\times n_k\times r_k}
$$

へ reshape して作ります。

そして、

$$
H_k
=
R_k^TQ_k^T
$$

なので、$R_k^T$ を左隣 core の右 bond へ吸収します。

$$
\boxed{
\widehat G^{(k-1)}
=
G^{(k-1)}
\times_{\text{right bond}}
R_k^T
}
$$

この QR sweep では、特異値を意図的には捨てません。

したがって、表現するテンソルは変えません。


## 5. Reduced QR で shape が変わる場合

通常、

$$
r_{k-1}
\le
n_kr_k
$$

なら、

$$
q_k=r_{k-1}
$$

なので bond shape は変わりません。

一方、

$$
r_{k-1}
>
n_kr_k
$$

なら reduced QR の economy-size により、

$$
q_k=n_kr_k<r_{k-1}
$$

となり、bond shape が機械的に縮むことがあります。

これは後段の SVD truncation のような、

> 小さい特異値を誤差と引き換えに捨てる

操作ではありません。

その bond が局所 shape 上そもそも持てない冗長な自由度を整理しているだけで、QR factorization 自体は exact です。

また、通常の `torch.linalg.qr(..., mode="reduced")` は **数値 rank を自動判定して削減する rank-revealing QR ではありません**。

近似を伴う rank 削減は、後段の truncated SVD で明示的に行います。


## 6. Right-Canonical 化すると何が嬉しいか

QR sweep 後、

$$
G^{(2)},\ldots,G^{(d)}
$$

が right-canonical なら、右側の環境は等長写像になります。

特に orthogonality center が第1 core にあるので、

$$
\boxed{
\|\mathcal X\|_F
=
\|G^{(1)}\|_F
}
$$

です。

したがって、全体相対誤差、

$$
\varepsilon
$$

から絶対誤差予算を作るとき、dense tensor を復元せず、

$$
\|\mathcal X\|_F
$$

を第1 core だけから計算できます。

TT-rounding が大規模 TT でも使えるために重要な性質です。


## 7. 全体誤差予算から局所予算を作る

ユーザー指定の全体相対誤差を、

$$
\varepsilon
$$

とします。

全体の絶対誤差予算は、

$$
\varepsilon
\|\mathcal X\|_F
$$

です。

$d-1$ 個の bond へ均等に二乗誤差予算を配るため、

$$
\boxed{
\delta
=
\frac{
\varepsilon\|\mathcal X\|_F
}{
\sqrt{d-1}
}
}
$$

とします。

各 bond の実際の切り捨て誤差を $e_k$ として、

$$
e_k
\le
\delta
$$

を満たせば、

$$
\sum_{k=1}^{d-1}
e_k^2
\le
(d-1)\delta^2.
$$

したがって、

$$
\sum_{k=1}^{d-1}
e_k^2
\le
\varepsilon^2\|\mathcal X\|_F^2
$$

となります。


## 8. Singular-Value Tail から保持 Rank を選ぶ

第 $k$ core の SVD を、

$$
V_k
=
U_k
\operatorname{diag}(S_k)
W_k^T
$$

とします。

特異値を、

$$
\sigma_{k,1}
\ge
\sigma_{k,2}
\ge
\cdots
\ge
\sigma_{k,q_k}
\ge0
$$

とします。

rank $r$ を残したときに捨てる二乗誤差は、

$$
e_k(r)^2
=
\sum_{j=r+1}^{q_k}
\sigma_{k,j}^2.
$$

局所予算 $\delta$ に対して、

$$
\boxed{
\widetilde r_k
=
\min
\left\{
r:
\sum_{j=r+1}^{q_k}
\sigma_{k,j}^2
\le
\delta^2
\right\}
}
$$

を選びます。

つまり、

> 小さい特異値から順番に、局所予算を超えない限り最大限捨てる

という操作です。

今回、

$$
e_k
=
\sqrt{
\sum_{j>\widetilde r_k}
\sigma_{k,j}^2
}
$$

を実際の局所誤差として記録します。


## 9. 浮動小数点比較の Slack

理論上、

$$
\text{tail\_sq}
\le
\delta^2
$$

なら切り捨て可能です。

しかし float64 でも丸め誤差があるため、判定だけには小さい slack を入れます。

dtype の machine epsilon を、

$$
u
$$

とし、

$$
\boxed{
\text{tail\_sq}
\le
\delta^2
+
100u
\max
\left(
\delta^2,
\sum_j\sigma_j^2
\right)
}
$$

を使います。

ただし diagnostics に保存する局所誤差は slack を含めません。

$$
\boxed{
e_k
=
\sqrt{\text{tail\_sq}}
}
$$

そのものを記録します。


## 10. Left-to-Right Truncated SVD Sweep

right-canonical 化した後、

$$
k=1,2,\ldots,d-1
$$

の順に処理します。

第 $k$ core を、

$$
V_k
=
\operatorname{reshape}
\left(
G^{(k)},
(r_{k-1}n_k,r_k)
\right)
$$

として、

$$
V_k
=
U_k\Sigma_kW_k^T
$$

と SVD します。

PyTorch では、

```python
U, S, Vh = torch.linalg.svd(V, full_matrices=False)
```

です。

保持 rank を $\widetilde r_k$ とすると、

$$
U_{\mathrm{keep}}
=
U(:,1:\widetilde r_k)
$$

を現在の core へ戻します。

$$
\widetilde G^{(k)}
=
\operatorname{reshape}
\left(
U_{\mathrm{keep}},
(r_{k-1},n_k,\widetilde r_k)
\right).
$$

残りは、

$$
B_k
=
\Sigma_{\mathrm{keep}}
W_{\mathrm{keep}}^T
\in
\mathbb R^{\widetilde r_k\times r_k}
$$

です。

これを右隣 core の左 bond へ吸収します。

$$
\boxed{
\widetilde G^{(k+1)}
(\beta_k,i_{k+1},\alpha_{k+1})
=
\sum_{\alpha_k}
B_k(\beta_k,\alpha_k)
G^{(k+1)}
(\alpha_k,i_{k+1},\alpha_{k+1})
}
$$


## 11. Canonical TT-rounding の誤差合成

right-canonical な環境を保ちながら左→右へ SVD truncation すると、各 bond で捨てた残差は直交的に蓄積します。

したがって、

$$
\boxed{
\|\mathcal X-\widetilde{\mathcal X}\|_F^2
=
\sum_{k=1}^{d-1}
e_k^2
}
$$

です。

さらに各 $e_k$ が、

$$
e_k\le\delta
$$

なので、

$$
\begin{aligned}
\|\mathcal X-\widetilde{\mathcal X}\|_F^2
&=
\sum_{k=1}^{d-1}e_k^2
\\
&\le
(d-1)\delta^2
\\
&=
\varepsilon^2
\|\mathcal X\|_F^2.
\end{aligned}
$$

よって、

$$
\boxed{
\|\mathcal X-\widetilde{\mathcal X}\|_F
\le
\varepsilon
\|\mathcal X\|_F
}
$$

です。

今回の小規模実験では、

$$
e_{\mathrm{theory}}
=
\sqrt{\sum_k e_k^2}
$$

と dense reconstruction から求めた実誤差を比較します。


## 12. PyTorch 設定


In [1]:
import math
from copy import deepcopy

import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

dtype = torch.float64
device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("dtype:", torch.get_default_dtype())
print("device:", device)


PyTorch: 2.11.0+cu128
dtype: torch.float64
device: cpu


## 13. 基本補助関数

以下は TT-rounding の新しいアルゴリズムそのものではなく、これまでの Notebook と同様に、

- core のコピー
- shape / rank 確認
- 小規模検証用 dense reconstruction
- Frobenius norm
- parameter 数

を扱う補助関数です。

`tt_to_dense` は今回の **テスト専用**です。

実際の大規模 TT-rounding で dense tensor を作ることは前提にしません。


In [7]:
from nn_compression.compression import (
    tt_num_parameters,
    tt_reconstruct as tt_to_dense,
)
from nn_compression.compression.tt_validation import validate_tt_cores


def clone_cores(cores):
    return [G.clone() for G in cores]


def tt_ranks(cores):
    return [int(G.shape[2]) for G in cores[:-1]]


def tt_physical_shape(cores):
    return tuple(int(G.shape[1]) for G in cores)


def fro_norm(X):
    return torch.linalg.vector_norm(X.reshape(-1))


## 14. 意図的に冗長な TT 入力を作る

単なる高 rank ランダム TT では、特異値が十分に減衰せず、許容誤差を与えても rank がほとんど下がらない場合があります。

そこで学習用には、

1. まず rank $(2,2)$ の TT を作る
2. それを rank $(4,3)$ の core shape へゼロ padding する
3. dense tensor は変えず、内部 bond だけ冗長にする

という入力を使います。

物理 shape は、

$$
(4,5,3)
$$

で固定です。

rounding 前後で物理 shape が変わらないことも確認します。


In [6]:
# 真の小さい TT-rank を持つ base TT
G1_base = torch.randn(1, 4, 2, dtype=dtype, device=device)
G2_base = torch.randn(2, 5, 2, dtype=dtype, device=device)
G3_base = torch.randn(2, 3, 1, dtype=dtype, device=device)

# bond shape だけを冗長に拡張: (2, 2) -> (4, 3)
G1 = torch.zeros(1, 4, 4, dtype=dtype, device=device)
G2 = torch.zeros(4, 5, 3, dtype=dtype, device=device)
G3 = torch.zeros(3, 3, 1, dtype=dtype, device=device)

G1[:, :, :2] = G1_base
G2[:2, :, :2] = G2_base
G3[:2, :, :] = G3_base

cores0 = [G1, G2, G3]
validate_tt_cores(cores0)

print("physical shape:", tt_physical_shape(cores0))
print("TT ranks:", tt_ranks(cores0))
print("core shapes:", [tuple(G.shape) for G in cores0])
print("TT parameters:", tt_num_parameters(cores0))


physical shape: (4, 5, 3)
TT ranks: [4, 3]
core shapes: [(1, 4, 4), (4, 5, 3), (3, 3, 1)]
TT parameters: 85


## 15. Dense Tensor と入力 TT を保存する

以後、rounding の各段階で core を更新します。

元の入力を壊さないよう、

$$
\text{cores\_original}
$$

を別に保存します。

dense tensor は小規模テストでだけ使います。


In [4]:
cores_original = clone_cores(cores0)

X_original = tt_to_dense(cores_original)
norm_original = fro_norm(X_original)

print("dense shape:", tuple(X_original.shape))
print("||X||_F:", norm_original.item())
print("input ranks:", tt_ranks(cores_original))
print("input parameters:", tt_num_parameters(cores_original))


dense shape: (4, 5, 3)
||X||_F: 14.773489486620214
input ranks: [4, 3]
input parameters: 85


## 16. Right-Orthogonality の確認関数

第 $k$ core を、

$$
H_k
=
\operatorname{reshape}
(G^{(k)},(r_{k-1},n_kr_k))
$$

としたとき、

$$
H_kH_k^T
\approx
I
$$

を確認します。

この確認は Notebook 05 で学んだ right-orthogonality と同じです。


In [5]:
def right_orthogonality_error(core):
    r_left, n_k, r_right = core.shape

    H = core.reshape(
        r_left,
        n_k * r_right,
    )

    gram = H @ H.mT

    I = torch.eye(
        r_left,
        dtype=core.dtype,
        device=core.device,
    )

    return fro_norm(gram - I)


def check_right_canonical(cores):
    # 第1 core は orthogonality center なので、
    # right-canonical 性の検査対象は第2 core以降。
    return [
        right_orthogonality_error(cores[k])
        for k in range(1, len(cores))
    ]


## 17. 演習1 — Right-to-Left QR Sweep

### 目的

既存 TT cores を、表現するテンソルを変えずに right-canonical form へ変換します。

処理順は、

$$
k=d,d-1,\ldots,2
$$

です。

### 1ステップで行うこと

1. 現在 core を
   $$
   H_k
   \in
   \mathbb R^{r_{k-1}\times(n_kr_k)}
   $$
   へ reshape
2. 
   $$
   H_k^T=Q_kR_k
   $$
   を reduced QR
3. `Q.mT` を新しい current core へ戻す
4. `R.mT` を左隣 core の右 bond に吸収する

### TODO

`right_canonicalize(cores)` を実装してください。

### 注意

`Q.shape[1]` は必ずしも元の `r_left` と同じとは限りません。

したがって、新しい左 bond size は、

```python
new_left_rank = Q.shape[1]
```

から取得してください。

### 考えること

- なぜ `H` ではなく `H.mT` を QR するのか
- なぜ新しい core は `Q.mT` なのか
- `R.mT` を左隣へ吸収すると全テンソルが保存されるのはなぜか
- QR sweep では、どこにも singular-value truncation がないこと


In [10]:
# TODO 1:
# right-to-left QR sweep を実装してください。
#
def right_canonicalize(cores: list[torch.Tensor]) -> list[torch.Tensor]:
    """TT core 列を、表現するテンソルを変えずに right-canonical form へ変換する。

    cores:
        入力 TT cores。各要素は ``(r_{k-1}, n_k, r_k)``。
        このリストと中の Tensor は変更しない。
    戻り値:
        新しい right-canonical core 列。左端 bond は 1、右端 bond は 1。
    """
    # - 入力 cores を直接変更しない
    cores_work = [core.clone() for core in cores]

    # cores[k] を右展開 → H.mT を QR → Q.mT を戻す → R.mT を cores[k-1] へ吸収
    for k in range(len(cores_work) - 1, 0, -1):  # k = d-1, ..., 1
        r_left, n_k, r_right = cores_work[k].shape
        H = cores_work[k].reshape(r_left, n_k * r_right)  # (r_{k-1}, n_k r_k)

        # - 2次元転置は .mT
        Q, R = torch.linalg.qr(H.mT, mode="reduced")  # H.mT = Q R
        new_left_rank = Q.shape[1]

        # 新しい第 k core（右直交）
        cores_work[k] = Q.mT.reshape(new_left_rank, n_k, r_right)

        # R.mT: (r_left, new_left_rank) を左隣の右 bond へ吸収
        cores_work[k - 1] = torch.tensordot(
            cores_work[k - 1],
            R.mT,
            dims=([-1], [0]),
        )

    return cores_work
    

## 18. 演習2 — QR Sweep の不変性・直交性・ノルム

### 目的

演習1で作った `right_canonicalize` を使い、次の3点を確認します。

### A. Dense tensor の不変性

$$
\boxed{
\mathcal X_{\mathrm{before}}
\approx
\mathcal X_{\mathrm{after}}
}
$$

### B. Right-canonical 性

第2 core 以降について、

$$
\boxed{
H_kH_k^T
\approx
I
}
$$

### C. Center norm

orthogonality center が第1 core にあるので、

$$
\boxed{
\|\mathcal X\|_F
\approx
\|G^{(1)}\|_F
}
$$

### TODO

1. `cores_rc = right_canonicalize(cores_original)` を作る
2. dense reconstruction を行う
3. QR前後の誤差を計算する
4. `check_right_canonical` で直交性誤差を調べる
5. dense norm と第1 core normを比較する

### 考えること

- QR sweep の reconstruction error は、truncation error か
- float64 でどの程度の誤差なら「不変」とみなせるか
- なぜ第1 core だけに全体ノルムを集められるのか


In [13]:
# TODO 2:
# right-canonicalization 後の不変性・直交性・ノルムを確認してください。
#
# --- 実験設定 ---
cores_rc = right_canonicalize(cores_original)

X_before = tt_to_dense(cores_original)
X_after = tt_to_dense(cores_rc)

norm_X = fro_norm(X_after)
norm_G1 = fro_norm(cores_rc[0])

# --- A. Dense tensor の不変性 ---
qr_reconstruction_error = fro_norm(X_before - X_after).item()

# --- B. Right-canonical 性（第2 core 以降） ---
right_orth_errors = check_right_canonical(cores_rc)
right_orth_errors_list = [e.item() for e in right_orth_errors]

# --- C. Center norm ---
center_norm_error = abs(norm_X.item() - norm_G1.item())

print("core shapes (before):", [tuple(G.shape) for G in cores_original])
print("core shapes (after) :", [tuple(G.shape) for G in cores_rc])
print("QR reconstruction error =", qr_reconstruction_error)
print("right-orthogonality errors =", right_orth_errors_list)
print("||X||_F     =", norm_X.item())
print("||G^(1)||_F =", norm_G1.item())
print("||X||_F - ||G^(1)||_F =", center_norm_error)


core shapes (before): [(1, 4, 4), (4, 5, 3), (3, 3, 1)]
core shapes (after) : [(1, 4, 4), (4, 5, 3), (3, 3, 1)]
QR reconstruction error = 3.395438385995374e-15
right-orthogonality errors = [2.7633878174254497e-16, 6.138747913423791e-16]
||X||_F     = 14.77348948662021
||G^(1)||_F = 14.773489486620212
||X||_F - ||G^(1)||_F = 1.7763568394002505e-15


## 19. 演習3 — `choose_rank` を単独で実装する

### 目的

TT-rounding からいったん切り離して、

> 特異値と局所二乗誤差予算から、保持 rank を決める

処理だけを実装します。

入力は、

- `s`：降順の特異値
- `budget_sq`：$\delta^2$
- `min_rank=1`

です。

返したいものは、

- `new_rank`
- `tail_sq`
- `local_error`
- `discard_count`
- `slack`

です。

### 数値例

$$
s=(5,\;2,\;0.6,\;0.3)
$$

として、

$$
\delta=0.7
$$

なら、

$$
0.6^2+0.3^2
=
0.45
<
0.49
=
0.7^2
$$

なので、後ろ2個を捨てられます。

期待する保持 rank は、

$$
\widetilde r=2
$$

です。

### TODO

1. 小さい特異値から二乗値を追加する
2. slack を含めて予算判定する
3. 予算を超える直前で止める
4. `min_rank` 未満にはしない
5. 実際の `tail_sq` には slack を加えない

### 考えること

- なぜ前からではなく末尾から捨てるのか
- `tail_sq` と `budget_sq` の単位は何か
- `local_error` はなぜ平方根を取るのか


In [ ]:
# TODO 3:
# choose_rank を実装してください。
#
# def choose_rank(
#     s,
#     budget_sq,
#     *,
#     min_rank=1,
#     slack_factor=100.0,
# ):
#     ...
#
# 判定:
# tail_sq <= budget_sq + slack
#
# slack:
# 100 * machine_epsilon * max(budget_sq, sum(s^2))


## 20. 演習4 — `choose_rank` の数値例

### 目的

演習3の関数を、

$$
s=(5,2,0.6,0.3),
\qquad
\delta=0.7
$$

で確認します。

期待値は、

$$
\widetilde r=2,
$$

$$
e^2
=
0.6^2+0.3^2
=
0.45,
$$

$$
e
=
\sqrt{0.45}
\approx
0.6708.
$$

### TODO

`choose_rank` を呼び、次を確認してください。

- `new_rank == 2`
- `tail_sq ≈ 0.45`
- `local_error ≈ sqrt(0.45)`
- `discard_count == 2`


In [ ]:
# TODO 4:
# choose_rank の単体テストを行ってください。
#
# s = torch.tensor([5.0, 2.0, 0.6, 0.3])
# delta = 0.7
#
# result = choose_rank(...)
#
# print と assertion で期待値を確認してください。


## 21. 局所誤差予算を計算する

right-canonicalization 後は、

$$
\|\mathcal X\|_F
=
\|G^{(1)}\|_F
$$

なので、dense tensor を使わず、

$$
\boxed{
\delta
=
\frac{
\varepsilon
\|G^{(1)}\|_F
}{
\sqrt{d-1}
}
}
$$

と計算できます。

コードでは、

- `eps_rel` $\leftrightarrow \varepsilon$
- `delta` $\leftrightarrow \delta$

です。

最初は、

$$
\varepsilon=10^{-8}
$$

程度の小さい値で、ゼロ padding による冗長な rank が落ちるか確認します。


In [ ]:
# このセルは、演習1・2で cores_rc を作った後に実行します。

eps_rel = 1e-8

# TODO:
# cores_rc の第1 coreだけから ||X||_F を求め、
# delta = eps_rel * ||X||_F / sqrt(d - 1)
# を計算してください。
#
# print:
# - ||X||_F
# - eps_rel
# - delta
# - delta**2


## 22. 演習5 — 1 Bond 分の SVD Truncation を観察する

### 目的

いきなり sweep 全体を関数化せず、最初の bond だけを手で追います。

Python の、

```python
k = 0
```

は理論の第1 core / 第1 bond に対応します。

### TODO

1. `cores_rc` をコピーする
2. 第1 core を
   $$
   V_1
   \in
   \mathbb R^{n_1\times r_1}
   $$
   へ reshape
3. 
   ```python
   U, S, Vh = torch.linalg.svd(
       V,
       full_matrices=False,
   )
   ```
   を実行
4. `choose_rank(S, delta**2)` で保持 rank を選ぶ
5. `U[:, :new_rank]` を第1 core に戻す
6. 
   $$
   \Sigma_{\mathrm{keep}}V_{\mathrm{keep}}^T
   $$
   に対応する行列を作る
7. その行列を第2 core の左 bond に吸収する
8. core shapes がどう変化したか確認する

### Shape で確認するもの

$$
V_1:
(n_1,r_1)
$$

$$
U_{\mathrm{keep}}:
(n_1,\widetilde r_1)
$$

$$
B_1:
(\widetilde r_1,r_1)
$$

$$
G_{\mathrm{new}}^{(2)}:
(\widetilde r_1,n_2,r_2)
$$

### 考えること

- QR sweep と違い、どこで近似が発生したか
- `S[:, None] * Vh` が何を表しているか
- 右隣への吸収で physical dimension が変わらないこと


In [ ]:
# TODO 5:
# left-to-right truncation の1ステップだけを実装してください。
#
# work = clone_cores(cores_rc)
# k = 0
#
# 確認:
# - V, U, S, Vh のshape
# - singular values
# - new_rank
# - local_error
# - 更新後 core shapes


## 23. 演習6 — Left-to-Right Truncation Sweep

### 目的

1 bond で確認した操作を、

$$
k=1,\ldots,d-1
$$

へ一般化します。

### 実装する関数

```python
left_to_right_truncate(
    cores_rc,
    eps_rel,
)
```

### 戻り値

1. rounded cores
2. diagnostics

diagnostics には少なくとも、

- `norm_x`
- `eps_requested`
- `delta`
- `local_errors`
- `local_tail_sq`
- `output_ranks`
- `global_error_estimate`
- `relative_error_estimate`
- `rank_details`

を保存します。

ここで、

$$
\text{global\_error\_estimate}
=
\sqrt{
\sum_k e_k^2
}
$$

です。

### 境界ケース

#### $\varepsilon=0$

intentional truncation を行いません。

right-canonicalized TT をそのまま返します。

#### Zero tensor

$$
\|\mathcal X\|_F=0
$$

なら、各 core shape を、

$$
(1,n_k,1)
$$

とする rank-1 zero TT を返します。

#### $d=1$

内部 bond がないので truncation はありません。

### TODO

関数本体を実装してください。

### 考えること

- `delta` を毎 bond で変える必要があるか
- 各局所 `tail_sq` の和が何を表すか
- `eps_rel=0` を `choose_rank` にそのまま渡すのではなく、なぜ特別扱いするか


In [ ]:
# TODO 6:
# left_to_right_truncate を実装してください。
#
# def left_to_right_truncate(cores_rc, eps_rel):
#     ...
#
# 必須:
# - torch.linalg.svd(..., full_matrices=False)
# - 2次元転置は .mT
# - 右隣への吸収は torch.einsum
# - choose_rank を使用
# - local tail energy を diagnostics に保存
# - eps_rel == 0
# - zero tensor
# - d == 1


## 24. 演習7 — `tt_round` を統合する

### 目的

ここまでの2段階、

$$
\text{right-canonicalize}
\rightarrow
\text{left-to-right truncate}
$$

を1つにまとめます。

API は、

```python
rounded_cores, info = tt_round(
    cores,
    eps_rel,
)
```

とします。

### TODO

1. 入力 validation
2. `eps_rel < 0` を拒否
3. input ranks を記録
4. right-canonicalization
5. left-to-right truncation
6. output ranks を記録
7. rounded cores と diagnostics を返す

### 重要

入力 `cores` 自体は変更しないでください。


In [ ]:
# TODO 7:
# tt_round を実装してください。
#
# def tt_round(cores, eps_rel):
#     ...


## 25. 演習8 — Dense Reconstruction で最終誤差を検証する

### 目的

最終的に、

$$
\widetilde{\mathcal X}
$$

を dense に戻し、

$$
e_{\mathrm{dense}}
=
\|\mathcal X-\widetilde{\mathcal X}\|_F
$$

を測定します。

一方、diagnostics に保存した局所誤差から、

$$
e_{\mathrm{theory}}
=
\sqrt{
\sum_{k=1}^{d-1}
e_k^2
}
$$

を求めます。

確認する式は、

$$
\boxed{
e_{\mathrm{dense}}
\approx
e_{\mathrm{theory}}
}
$$

および、

$$
\boxed{
e_{\mathrm{dense}}
\le
\varepsilon
\|\mathcal X\|_F
+
\text{numerical tolerance}
}
$$

です。

### TODO

1. `tt_round` を実行
2. input / output ranks を表示
3. input / output parameters を表示
4. rounded tensor を dense reconstruction
5. dense absolute error
6. dense relative error
7. theory error
8. requested bound
9. dense error と theory error の差
10. assertion

### 考えること

- physical shape は変化したか
- TT-rank はどう変化したか
- parameter 数はどう変化したか
- 今回の小さい例では dense parameter 数よりTTが多い可能性があるが、それは失敗か


In [ ]:
# TODO 8:
# TT-rounding 全体を実行し、dense reconstruction で検証してください。
#
# 表示:
# - input ranks
# - output ranks
# - input parameters
# - output parameters
# - dense shape before / after
# - dense error
# - dense relative error
# - theory error
# - requested error bound
#
# assertion:
# dense_error <= eps_rel * ||X||_F + tolerance


## 26. 演習9 — $\varepsilon$ Sweep

### 目的

全体相対誤差、

$$
\varepsilon
$$

を変えたとき、

- 出力 TT-rank
- parameter 数
- theory error
- dense error
- relative error

がどう変化するか確認します。

試す値は、

$$
\varepsilon
\in
\{
0,\;
10^{-12},\;
10^{-8},\;
10^{-4},\;
10^{-2},\;
10^{-1}
\}
$$

程度で十分です。

### 期待する傾向

$$
\varepsilon
\uparrow
\Rightarrow
\delta
\uparrow
\Rightarrow
\text{捨てられる特異値が増える}
\Rightarrow
\text{rankが下がりやすい}
$$

です。

ただし、rank は連続量ではないため、$\varepsilon$ を少し変えても rank が変化しない区間があります。

### TODO

結果を表形式で整理してください。


In [ ]:
# TODO 9:
# epsilon sweep を実行してください。
#
# eps_values = [
#     0.0,
#     1e-12,
#     1e-8,
#     1e-4,
#     1e-2,
#     1e-1,
# ]
#
# 各 eps について:
# - output ranks
# - output parameters
# - theory error
# - dense error
# - relative error
#
# を記録してください。


## 27. 演習10 — Zero Tensor と $d=1$

### A. Zero tensor

物理 shape を例えば、

$$
(4,5,3)
$$

とした rank-1 zero TT を用意します。

$$
\mathcal X=0.
$$

$\varepsilon>0$ で `tt_round` し、

- 出力が zero tensor
- 全 core rank が1
- dense error が0

になることを確認します。

### B. $d=1$

$$
G^{(1)}
\in
\mathbb R^{1\times n_1\times1}
$$

だけを持つ TT を作ります。

内部 bond が存在しないので、

- QR sweepなし
- SVD truncationなし
- tensor不変

になることを確認します。

### TODO

2ケースを独立にテストしてください。


In [ ]:
# TODO 10:
# 1. zero TT
# 2. d = 1 TT
#
# の境界ケースを検証してください。


## 28. 理論記号とコード変数の対応

この Notebook では、次の対応を固定します。

| 理論 | コード | 意味 |
| --- | --- | --- |
| $\varepsilon$ | `eps_rel` | 全体相対誤差 |
| $\delta$ | `delta` | 1 bond あたりの局所絶対誤差予算 |
| $(\sigma_{k,j})$ | `S` | 第 $k$ bond の特異値 |
| $e_k^2$ | `tail_sq` | 実際に捨てた特異値二乗和 |
| $e_k$ | `local_error` | 第 $k$ bond の局所誤差 |
| $\widetilde r_k$ | `new_rank` | truncation 後の bond rank |
| $\sqrt{\sum_ke_k^2}$ | `global_error_estimate` | 局所誤差から計算した全体誤差 |
| $(r_1,\ldots,r_{d-1})$ | `tt_ranks(cores)` | TT-rank |

特に、

$$
\delta
$$

と、

$$
e_k
$$

を混同しないことが重要です。

$\delta$ は **許される予算**、

$e_k$ は **実際に捨てた量**です。

したがって通常、

$$
e_k
\le
\delta
$$

です。


## 29. 今回のまとめ

TT-rounding は、

> 既存の TT cores を入力として、dense tensor を復元せずに内部 rank を再圧縮する

アルゴリズムです。

### Step 1：Right-to-Left QR Sweep

$$
H_k^T
=
Q_kR_k
$$

として、

$$
Q_k^T
$$

を right-canonical core にし、

$$
R_k^T
$$

を左隣へ吸収します。

この段階は exact であり、意図的な truncation はありません。

### Step 2：局所誤差予算

$$
\boxed{
\delta
=
\frac{
\varepsilon\|\mathcal X\|_F
}{
\sqrt{d-1}
}
}
$$

です。

right-canonical TT では、

$$
\|\mathcal X\|_F
=
\|G^{(1)}\|_F
$$

なので dense tensor は不要です。

### Step 3：Left-to-Right SVD Truncation

$$
V_k
=
U_k\Sigma_kW_k^T
$$

から、

$$
\sum_{j>\widetilde r_k}
\sigma_{k,j}^2
\le
\delta^2
$$

を満たす最小の $\widetilde r_k$ を選びます。

### Step 4：誤差検証

各 bond で、

$$
e_k^2
=
\sum_{j>\widetilde r_k}
\sigma_{k,j}^2
$$

を記録し、

$$
\boxed{
\|\mathcal X-\widetilde{\mathcal X}\|_F^2
=
\sum_{k=1}^{d-1}
e_k^2
}
$$

を小規模 dense reconstruction で確認します。

最終的に、

$$
\boxed{
\|\mathcal X-\widetilde{\mathcal X}\|_F
\le
\varepsilon
\|\mathcal X\|_F
}
$$

が今回の到達目標です。

次の段階では、TT 演算によって rank が増えた具体例に対して rounding を使い、なぜ実務で再圧縮が必要になるのかを確認できます。
